# Review — 2_openai (100% local, with Ollama)

Review notebook of the **design patterns** covered in `1_lab1` → `4_lab4` (OpenAI Agents SDK),
rewritten to run entirely on **free Ollama models** (no OpenAI key, no tracing upload).

## Setup

```bash
ollama pull llama3.2   # small model for plain chat
ollama pull qwen3:8b   # reliable tool-calling / structured-output model
ollama serve           # if not already running
pip install openai-agents ddgs   # ddgs = free web search, replaces the paid WebSearchTool
```

> **What changes vs. the labs?** Only *where the model lives*. The SDK talks to any
> OpenAI-compatible endpoint via `OpenAIChatCompletionsModel` (lab3). Everything else
> (`Agent`, `Runner`, tools, handoffs, guardrails) is identical.
> Tracing goes to platform.openai.com, so it is disabled here.

## Index of macro-blocks

| # | Block | Pattern | Source lab |
|---|-------|---------|------------|
| 1 | Agent + Runner + streaming | Agent = name + instructions + model; `Runner` = agent loop | lab1 |
| 2 | Function tools | `@function_tool` builds the JSON schema from signature + docstring | lab1 |
| 3 | Memory | Manual `to_input_list()` vs `SQLiteSession` | lab1 |
| 4 | Orchestration by code | `asyncio.gather` in parallel + picker agent | lab2 |
| 5 | Orchestration by LLM: agents as tools | Manager calls agents like tools (A→B→A) | lab2 |
| 6 | Orchestration by LLM: handoffs | Control passes across (A→B) | lab2 |
| 7 | Any model | `AsyncOpenAI(base_url)` + `OpenAIChatCompletionsModel` | lab3 |
| 8 | Structured outputs | `output_type=PydanticModel` | lab3 / lab4 |
| 9 | Guardrails | Framework `@output_guardrail` vs plain `if` | lab3 |
| 10 | Deep research pipeline | Orchestrate by code: planner → parallel search → writer → emailer | lab4 |

Skipped (need OpenAI cloud / extra infra): hosted `WebSearchTool`, `SandboxAgent`, MCP teaser.

In [ ]:
import asyncio, json, requests
from openai import AsyncOpenAI
from pydantic import BaseModel, Field
from IPython.display import Markdown, display
from openai.types.responses import ResponseTextDeltaEvent
from agents import (
    Agent, Runner, function_tool, ModelSettings, SQLiteSession, OpenAIChatCompletionsModel,
    output_guardrail, GuardrailFunctionOutput, set_tracing_disabled,
)

set_tracing_disabled(True)  # tracing would try to upload to platform.openai.com

ollama = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
CHAT_MODEL = OpenAIChatCompletionsModel(model="llama3.2", openai_client=ollama)   # plain chat
TOOL_MODEL = OpenAIChatCompletionsModel(model="qwen3:8b", openai_client=ollama)   # tools + structured output

def check_model(name):
    try:
        models = {m["id"] for m in requests.get("http://localhost:11434/v1/models", timeout=3).json().get("data", [])}
    except Exception:
        print("Ollama unreachable on localhost:11434 — make sure 'ollama serve' is running.")
        return False
    if not any(m == name or m.startswith(name + ":") for m in models):
        print(f"Model '{name}' not found. Run: ollama pull {name}")
        return False
    return True

check_model("llama3.2") and check_model("qwen3:8b")

## Block 1 — Agent, Runner (agent loop), streaming

**Pattern:** an `Agent` is just *name + instructions + model (+ tools)*. `Runner.run(agent, input)`
is the **agent loop**: it calls the model, executes tools, and repeats until there is a final output.
`Runner.run_streamed` yields events so you can print tokens as they arrive.

(Notebooks already run an event loop, so we `await` directly instead of `asyncio.run`.)

In [ ]:
agent = Agent(name="Jokester", instructions="You are a joke teller", model=CHAT_MODEL)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

In [ ]:
# Detail of the LLM calls (the messages that actually went through the loop)
result.to_input_list()

In [ ]:
# Streaming
result = Runner.run_streamed(agent, input="Please tell me 3 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

## Block 2 — Function tools

**Pattern:** decorate a plain Python function with `@function_tool`. The SDK builds the JSON
schema from the **type hints** and the **docstring** — no hand-written schema (compare with the
manual `*_json` dicts in `1_foundations`). The agent loop also does the dispatch for you.

Here the Pushover side-effect is replaced by a `print` to stay 100% local.

In [ ]:
@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    print(f"[PUSH] {message}")
    return "Push sent"

push_tool.params_json_schema   # auto-generated

In [ ]:
notifier = Agent(name="Notifier", model=TOOL_MODEL, instructions="You notify the user upon request", tools=[push_tool])
result = await Runner.run(notifier, "Notify the user that the pizza is here")
print(result.final_output)

## Block 3 — Memory

**Pattern:** each `Runner.run()` is stateless — same *illusion of memory* as in `1_foundations`.
Two fixes:

1. **Manual:** feed `result.to_input_list() + [new_message]` back in.
2. **`SQLiteSession`:** the SDK stores and replays history for you (in-memory by default;
   `SQLiteSession("id", "memory.db")` to persist on disk).

In [ ]:
assistant = Agent(name="Assistant", model=CHAT_MODEL)

r = await Runner.run(assistant, "Hi there. My name is Giulio.")
r2 = await Runner.run(assistant, "What's my name?")
print("No memory:    ", r2.final_output)

# Approach 1: manual
next_input = r.to_input_list() + [{"role": "user", "content": "What's my name?"}]
r3 = await Runner.run(assistant, next_input)
print("Manual list:  ", r3.final_output)

# Approach 2: session
session = SQLiteSession("review-session")
await Runner.run(assistant, "Hi there. My name is Giulio.", session=session)
r4 = await Runner.run(assistant, "What's my name?", session=session)
print("SQLiteSession:", r4.final_output)

## Block 4 — Orchestration by code (deterministic)

**Pattern:** *you* write the workflow in Python: run 3 agents **in parallel** with
`asyncio.gather`, then hand all outputs to a **picker** agent. Predictable, debuggable, cheap.

Ollama serves requests one at a time by default, so "parallel" is concurrent but
may be queued by the server — the pattern is what matters.

In [ ]:
intro = """
You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails. Keep them under 100 words.
"""
styles = {
    "Professional": "Your email style is professional, serious, with gravitas and credibility.",
    "Humorous":     "Your email style is witty, engaging, and humorous.",
    "Executive":    "Your email style is concise, to the point, in the style of a busy senior executive.",
}
sales_agents = [Agent(name=f"{n} Sales Agent", instructions=intro + s, model=TOOL_MODEL) for n, s in styles.items()]

sales_picker = Agent(
    name="Sales Picker", model=TOOL_MODEL,
    instructions="You pick the best cold sales email from the given options. Imagine you are a customer and pick the one "
                 "you are most likely to respond to. Do not give an explanation; reply with the selected email only.",
)

message = "Write a cold sales email"
results = await asyncio.gather(*(Runner.run(a, message) for a in sales_agents))
outputs = [r.final_output for r in results]

emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)
best = await Runner.run(sales_picker, emails)
display(Markdown(f"**Best email:**\n\n{best.final_output}"))

## Block 5 — Orchestration by LLM: agents as tools

**Pattern:** `agent.as_tool(name, description)` turns an agent into a tool. A **manager**
(planning agent) decides *when* to call each one, and control **returns** to the manager: `A → B → A`.
More powerful than code orchestration, less predictable — smaller models need prompt iteration.

In [ ]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects

    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    print(f"[EMAIL] {subject}\n{text_body}")   # replaces SMTP / Pushover
    return "Email sent successfully"

description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."
writer_tools = [a.as_tool(tool_name=f"sales_email_writer_{i+1}", tool_description=description) for i, a in enumerate(sales_agents)]

manager_instructions = "You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_email_writer tools."
task = """
Follow these steps:
1. Generate Drafts: use each of the three sales_email_writer tools to generate different email drafts.
   Do not proceed until all three drafts are ready, one from each tool.
2. Evaluate and Select: choose the single best email.
3. Use your send_email_tool to send the best email (and only the best email). Only send 1 email.
"""
sales_manager = Agent(name="Sales Manager", instructions=manager_instructions, tools=writer_tools + [send_email_tool], model=TOOL_MODEL)

result = await Runner.run(sales_manager, task)
print(result.final_output)

## Block 6 — Orchestration by LLM: handoffs

**Pattern:** `handoffs=[other_agent]` lets an agent **delegate and pass control across**: `A → B`
(the final answer comes from B). Under the hood a handoff is just another tool call.
Handy for triage/routing; for planner-style flows, agents-as-tools (Block 5) is usually more reliable.
`tool_choice="required"` forces the sender to actually use its tool.

In [ ]:
sales_sender = Agent(
    name="Sales Sender", model=TOOL_MODEL,
    instructions="You pick the best cold sales email from the given options and send it with your tool.",
    tools=[send_email_tool], model_settings=ModelSettings(tool_choice="required"),
)

manager = Agent(
    name="Sales Manager", model=TOOL_MODEL,
    instructions="You are a Sales Manager at ComplAI. You get your sales team to draft emails, then hand off to the sales sender.",
    tools=writer_tools, handoffs=[sales_sender],
)
result = await Runner.run(manager, """
1. Use each of the three sales_email_writer tools to generate a draft. Wait for all three.
2. Hand off to the Sales Sender to choose and send the best email.
""")
print(result.final_output)

## Block 7 — Any model, any provider

**Pattern (3 steps, straight from lab3):**

1. find the provider's OpenAI-compatible `base_url`
2. create an `AsyncOpenAI` client
3. wrap it in `OpenAIChatCompletionsModel(model=..., openai_client=...)`

We already did it in the setup cell (Ollama). Below, two *different* local models play two
agents, and are used side by side. To use paid providers just change `base_url` / `api_key`:

| Provider | `base_url` |
|---|---|
| Gemini | `https://generativelanguage.googleapis.com/v1beta/openai/` |
| OpenRouter | `https://openrouter.ai/api/v1` |
| Groq | `https://api.groq.com/openai/v1` |

In [ ]:
for name, model in [("llama3.2", CHAT_MODEL), ("qwen3:8b", TOOL_MODEL)]:
    r = await Runner.run(Agent(name=name, instructions="Answer in one sentence.", model=model), "What is an AI agent?")
    display(Markdown(f"**{name}**: {r.final_output}"))

## Block 8 — Structured outputs

**Pattern:** define a Pydantic `BaseModel`, pass it as `output_type`. The SDK sends the JSON schema,
the model answers in JSON, and you get back a **Python object** (`result.final_output`) — not text to parse.
`Field(description=...)` is part of the prompt: describe each field well.

In [ ]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

checker = Agent(name="Checker", instructions="You review potential sales emails", model=TOOL_MODEL, output_type=EmailReview)

bad_email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Ed
"""
review = (await Runner.run(checker, bad_email)).final_output
print(type(review).__name__, review)

## Block 9 — Guardrails

**Pattern:** controls, coded or LLM-based, that block undesirable behaviour. Two ways:

1. **Framework** `@output_guardrail`: runs a checker agent on the final output; raises
   `OutputGuardrailTripwireTriggered` if `tripwire_triggered=True`.
   *Gotcha:* input guardrails run only on the **first** agent's input, output guardrails only on the **last** agent's output.
2. **Plain code:** a separate `Runner.run(checker, ...)` + `if`. Simpler, works in any framework.

In [ ]:
from agents import OutputGuardrailTripwireTriggered

@output_guardrail
async def email_guardrail(ctx, agent, message):
    review = (await Runner.run(checker, message, context=ctx.context)).final_output
    return GuardrailFunctionOutput(
        output_info={"review": review},
        tripwire_triggered=review.contains_placeholders or not review.is_professional,
    )

cowboy_instructions = intro + "\nSpeak like a cowboy"
guarded = Agent(name="Cowboy", instructions=cowboy_instructions, model=TOOL_MODEL, output_guardrails=[email_guardrail])

try:
    print((await Runner.run(guarded, "Write a cold sales email")).final_output)
except OutputGuardrailTripwireTriggered as e:
    print("Blocked by guardrail:", e.guardrail_result.output.output_info)

In [ ]:
# The plain-code equivalent
email = (await Runner.run(Agent(name="Simple Cowboy", instructions=cowboy_instructions, model=TOOL_MODEL), "Write a cold sales email")).final_output
review = (await Runner.run(checker, email)).final_output
print("Email is good" if review.is_professional and not review.contains_placeholders else "Not professional or has placeholders: not sent")

## Block 10 — Deep research pipeline (the lab4 pattern)

**Pattern:** orchestrate **by code**, one `Runner.run()` per step, **structured output at every
hand-over** — the "bulletproof" way:

`Planner → (N searches in parallel) → Writer → Emailer`

Replacements to stay local: the hosted, paid `WebSearchTool` → a free `@function_tool` on DuckDuckGo (`ddgs`);
the SMTP email → a `print`.

In [ ]:
from ddgs import DDGS

HOW_MANY_SEARCHES = 3

@function_tool
def web_search(query: str) -> str:
    """ Search the web and return the top results (title, snippet) as text """
    hits = DDGS().text(query, max_results=5)
    return "\n".join(f"- {h['title']}: {h['body']}" for h in hits)

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")

search_agent = Agent(
    name="Search Agent", model=TOOL_MODEL, tools=[web_search], model_settings=ModelSettings(tool_choice="required"),
    instructions="You are a research assistant. Given a search term, search the web and produce a concise summary "
                 "(2-3 paragraphs, under 300 words). Reply only with the summary.",
)
planner_agent = Agent(
    name="Planner Agent", model=TOOL_MODEL, output_type=WebSearchPlan,
    instructions=f"You are a research assistant. Given a query, come up with {HOW_MANY_SEARCHES} web searches to best answer it.",
)
writer_agent = Agent(
    name="Writer Agent", model=TOOL_MODEL, output_type=ReportData,
    instructions="You are a senior researcher. Given the original query and summarized research, write a cohesive, "
                 "detailed report in markdown (at least 500 words).",
)
email_agent = Agent(
    name="Email Agent", model=TOOL_MODEL, tools=[send_email_tool],
    instructions="You are given a report. Use your tool to send it as a clean HTML email with an appropriate subject line.",
)

In [ ]:
async def search(item: WebSearchItem):
    return (await Runner.run(search_agent, f"Search term: {item.query}\nReason for searching: {item.reason}")).final_output

async def run_searches(query: str):
    plan = (await Runner.run(planner_agent, f"Query: {query}")).final_output
    print(f"Will perform {len(plan.searches)} searches")
    return await asyncio.gather(*(search(i) for i in plan.searches))

async def write_report(query: str, search_results: list[str]):
    return (await Runner.run(writer_agent, f"Original query: {query}\nSummarized search results: {search_results}")).final_output

query = "Most popular AI Agent frameworks in 2026"
search_results = await run_searches(query)
report = await write_report(query, search_results)
display(Markdown(report.markdown_report))
await Runner.run(email_agent, report.markdown_report);

## Pattern recap

| Pattern | Key idea |
|---|---|
| `Agent` + `Runner` | Agent = config; `Runner` = the `while` tool loop from `1_foundations`, built in |
| `@function_tool` | JSON schema from type hints + docstring; dispatch handled by the SDK |
| Memory | Stateless calls; `to_input_list()` or `SQLiteSession` replays history |
| Orchestration by code | Deterministic: `asyncio.gather` + explicit steps |
| Agents as tools | Manager calls agents, control returns (A→B→A) |
| Handoffs | Control passes across (A→B); a tool call under the hood |
| `OpenAIChatCompletionsModel` | Any OpenAI-compatible provider, incl. local Ollama |
| Structured outputs | `output_type=BaseModel` → Python object instead of text |
| Guardrails | Framework tripwire or, simpler, a checker agent + `if` |
| Pipeline with typed hand-overs | Planner → parallel search → writer → emailer |

To review the original paid version (OpenAI), compare with `1_lab1` → `4_lab4` in this folder.
For the from-scratch equivalents of these patterns, see `1_foundations/_review_1_foundations.ipynb`.